# Generation Evaluation

Evaluates the quality of RAG-generated answers against gold answers from the benchmark dataset.

**Metrics implemented:**

| Category | Metric | Notes |
|----------|--------|-------|
| Lexical overlap | Exact Match (EM) | Case/punctuation normalised |
| Lexical overlap | Token-level F1 | Whitespace-tokenised |
| Lexical overlap | ROUGE-1 / ROUGE-2 / ROUGE-L | Unigram, bigram, LCS |
| Semantic similarity | BERTScore (F1) | Optional — requires `bert-score` |

**Input:** `results/output_results.json` (or exploration folder)  
**Benchmark:** `../data/benchmark_dataset.json`  
**Output:** `results/generation_evaluation_summary.json`


In [ ]:
# Install required packages (run once; skip if already installed)
%pip install rouge-score bert-score nltk --quiet


In [1]:
import json
import os
import re
import string
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ROUGE (rouge-score)
try:
    from rouge_score import rouge_scorer
    ROUGE_AVAILABLE = True
except ImportError:
    ROUGE_AVAILABLE = False
    print("[WARN] rouge-score not installed — ROUGE metrics will be skipped.")
    print("       Run:  pip install rouge-score")

# BERTScore (optional — slow on CPU, best with GPU)
try:
    from bert_score import score as bert_score_fn
    BERTSCORE_AVAILABLE = True
except ImportError:
    BERTSCORE_AVAILABLE = False
    print("[INFO] bert-score not installed — BERTScore will be skipped (optional).")
    print("       Run:  pip install bert-score")

print(f"ROUGE available    : {ROUGE_AVAILABLE}")
print(f"BERTScore available: {BERTSCORE_AVAILABLE}")


ROUGE available    : True
BERTScore available: True


In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
BENCHMARK_PATH  = "../data/benchmark_dataset.json"

# Single output file to evaluate:
OUTPUT_PATH     = "results/output_results.json"

# OR evaluate all exploration outputs at once (set to None to skip):
EXPLORATION_DIR = "results/exploration"

EVAL_OUTPUT     = "results/generation_evaluation_summary.json"
os.makedirs("results", exist_ok=True)

# ── BERTScore model (only used if BERTSCORE_AVAILABLE) ───────────────────
BERTSCORE_MODEL = "distilbert-base-uncased"  # lighter than bert-base


## 1. Load Benchmark

In [7]:
def load_benchmark(path: str) -> Dict[str, dict]:
    """
    Loads the nested benchmark JSON:
      {sources: [{source, questions: [{question, answer}]}]}
    Returns dict keyed by query_id (q_001, q_002, ...).
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    benchmark = {}
    idx = 1
    for src_block in data["sources"]:
        source = src_block["source"]
        for item in src_block["questions"]:
            qid = f"{idx:d}"
            benchmark[qid] = {
                "query_id": qid,
                "query":    item["question"],
                "answer":   item["answer"],
                "source":   source,
            }
            idx += 1
    print(f"Loaded {len(benchmark)} benchmark entries from {path}")
    return benchmark


benchmark = load_benchmark(BENCHMARK_PATH)
# Preview
for qid, entry in list(benchmark.items())[:2]:
    print(f"  {qid}: {entry['query'][:60]}")
    print(f"       → {entry['answer'][:80]}")


Loaded 72 benchmark entries from ../data/benchmark_dataset.json
  1: What does the word 'bibimbap' literally mean in Korean?
       → The term bibim means mixing and bap is cooked rice, meaning 'cooked rice', so it
  2: What does the Japanese term 'nuuhara' mean?
       → Nuuhara is short for 'nuudoru harasumento' (noodle harassment), describing the l


## 2. Normalisation & Metric Helpers

In [8]:
def normalise(text: str) -> str:
    """
    Lowercase, remove articles, collapse whitespace, strip punctuation.
    Mirrors the normalisation used in SQuAD / TriviaQA evaluation.
    """
    text = text.lower()
    # Remove articles
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Collapse whitespace
    text = " ".join(text.split())
    return text


def tokenise(text: str) -> List[str]:
    return normalise(text).split()


# ── Exact Match ─────────────────────────────────────────────────────────────
def exact_match(prediction: str, reference: str) -> float:
    return float(normalise(prediction) == normalise(reference))


# ── Token-level F1 ──────────────────────────────────────────────────────────
def token_f1(prediction: str, reference: str) -> Tuple[float, float, float]:
    """Returns (f1, precision, recall)."""
    pred_tokens = tokenise(prediction)
    ref_tokens  = tokenise(reference)
    if not pred_tokens or not ref_tokens:
        return (1.0, 1.0, 1.0) if pred_tokens == ref_tokens else (0.0, 0.0, 0.0)
    common = Counter(pred_tokens) & Counter(ref_tokens)
    n_common = sum(common.values())
    if n_common == 0:
        return 0.0, 0.0, 0.0
    precision = n_common / len(pred_tokens)
    recall    = n_common / len(ref_tokens)
    f1        = 2 * precision * recall / (precision + recall)
    return f1, precision, recall


# ── ROUGE (via rouge-score library) ─────────────────────────────────────────
if ROUGE_AVAILABLE:
    _rouge_scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=False
    )

def compute_rouge(prediction: str, reference: str) -> Dict[str, float]:
    if not ROUGE_AVAILABLE:
        return {}
    scores = _rouge_scorer.score(reference, prediction)
    return {
        "rouge1_f": round(scores["rouge1"].fmeasure,  4),
        "rouge2_f": round(scores["rouge2"].fmeasure,  4),
        "rougeL_f": round(scores["rougeL"].fmeasure,  4),
        "rouge1_p": round(scores["rouge1"].precision,  4),
        "rouge1_r": round(scores["rouge1"].recall,     4),
    }


# ── Quick smoke-test ─────────────────────────────────────────────────────────
pred = "Bibimbap means mixed rice in Korean"
ref  = "Bibimbap literally means mixed rice: bibim means mixing and bap means cooked rice"
print("EM  :", exact_match(pred, ref))
f1, p, r = token_f1(pred, ref)
print(f"F1  : {f1:.4f}  P={p:.4f}  R={r:.4f}")
if ROUGE_AVAILABLE:
    print("ROUGE:", compute_rouge(pred, ref))


EM  : 0.0
F1  : 0.4211  P=0.6667  R=0.3077
ROUGE: {'rouge1_f': 0.4211, 'rouge2_f': 0.2353, 'rougeL_f': 0.4211, 'rouge1_p': 0.6667, 'rouge1_r': 0.3077}


## 3. Evaluation Functions

In [9]:
def evaluate_single(prediction: str, reference: str) -> dict:
    """Compute all metrics for one (prediction, reference) pair."""
    em = exact_match(prediction, reference)
    f1, prec, rec = token_f1(prediction, reference)
    result = {
        "exact_match": em,
        "f1":          round(f1,   4),
        "precision":   round(prec, 4),
        "recall":      round(rec,  4),
    }
    if ROUGE_AVAILABLE:
        result.update(compute_rouge(prediction, reference))
    return result


def evaluate_output(output_path: str, benchmark: Dict[str, dict]) -> dict:
    """
    Evaluate one generation output file against the benchmark.

    output_path : path to a JSON file produced by inference.ipynb
                  format: {results: [{query_id, query, response, retrieved_context}]}
    Returns a summary dict with per-query scores and aggregate metrics.
    """
    with open(output_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    results = data.get("results", [])

    per_query      = []
    by_source      = defaultdict(list)
    missing        = []

    # Collect scores
    for rec in results:
        qid = rec["query_id"]
        if qid not in benchmark:
            missing.append(qid)
            continue
        gold     = benchmark[qid]
        response = rec.get("response", "").strip()
        reference = gold["answer"].strip()

        scores = evaluate_single(response, reference)
        entry = {
            "query_id": qid,
            "query":    rec["query"],
            "response": response,
            "reference": reference,
            "source":   gold["source"],
            **scores,
        }
        per_query.append(entry)
        by_source[gold["source"]].append(scores)

    n = len(per_query)
    if n == 0:
        print(f"[WARN] No matching queries in {output_path}")
        return {}

    # ── Aggregate ────────────────────────────────────────────────────────────
    metric_keys = ["exact_match", "f1", "precision", "recall"]
    if ROUGE_AVAILABLE:
        metric_keys += ["rouge1_f", "rouge2_f", "rougeL_f"]

    aggregate = {k: round(sum(e[k] for e in per_query) / n, 4) for k in metric_keys}

    source_agg = {}
    for src, entries in by_source.items():
        ns = len(entries)
        source_agg[src] = {
            k: round(sum(e[k] for e in entries) / ns, 4)
            for k in metric_keys
        }
        source_agg[src]["n"] = ns

    summary = {
        "output_path":      output_path,
        "n_evaluated":      n,
        "n_in_benchmark":   len(benchmark),
        "n_missing":        len(missing),
        "aggregate":        aggregate,
        "by_source":        source_agg,
        "per_query":        per_query,
    }
    return summary


def print_summary(tag: str, summary: dict):
    agg = summary["aggregate"]
    print(f"\n{'='*80}")
    print(f"  {tag}")
    print(f"{'='*80}")
    print(f"  Queries evaluated : {summary['n_evaluated']} / {summary['n_in_benchmark']}")
    print(f"  Exact Match       : {agg['exact_match']:.4f}")
    print(f"  Token F1          : {agg['f1']:.4f}  (P={agg['precision']:.4f}  R={agg['recall']:.4f})")
    if ROUGE_AVAILABLE:
        print(f"  ROUGE-1 F         : {agg['rouge1_f']:.4f}")
        print(f"  ROUGE-2 F         : {agg['rouge2_f']:.4f}")
        print(f"  ROUGE-L F         : {agg['rougeL_f']:.4f}")
    print()
    print("  By source:")
    for src, stats in sorted(summary["by_source"].items()):
        print(f"    {src:40s} n={stats['n']:3d}  F1={stats['f1']:.4f}", end="")
        if ROUGE_AVAILABLE:
            print(f"  R1={stats['rouge1_f']:.4f}", end="")
        print()


## 4. Evaluate Single Output

In [10]:
# ── Evaluate the main output file ───────────────────────────────────────────
if os.path.exists(OUTPUT_PATH):
    summary_main = evaluate_output(OUTPUT_PATH, benchmark)
    print_summary("Main output", summary_main)
else:
    print(f"[SKIP] {OUTPUT_PATH} not found — run inference.ipynb first.")
    summary_main = None



  Main output
  Queries evaluated : 59 / 72
  Exact Match       : 0.0000
  Token F1          : 0.0740  (P=0.0517  R=0.1985)
  ROUGE-1 F         : 0.0908
  ROUGE-2 F         : 0.0048
  ROUGE-L F         : 0.0677

  By source:
    Around the World in 80 Cuisines Blog     n=  5  F1=0.1109  R1=0.1201
    Wikibooks                                n= 19  F1=0.0487  R1=0.0636
    Wikipedia                                n= 35  F1=0.0825  R1=0.1013


## 5. BERTScore (Optional — GPU recommended)

In [11]:
def add_bertscore(summary: dict, model_type: str = BERTSCORE_MODEL) -> dict:
    """
    Compute BERTScore for all queries in a summary dict and add to each entry.
    Modifies summary in-place and returns it.
    Requires bert-score to be installed.
    """
    if not BERTSCORE_AVAILABLE:
        print("[SKIP] bert-score not installed.")
        return summary

    preds = [e["response"]  for e in summary["per_query"]]
    refs  = [e["reference"] for e in summary["per_query"]]

    print(f"Computing BERTScore for {len(preds)} pairs (model={model_type})...")
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    P, R, F = bert_score_fn(preds, refs, model_type=model_type,
                             device=device, verbose=False)

    for i, entry in enumerate(summary["per_query"]):
        entry["bertscore_f"] = round(F[i].item(), 4)
        entry["bertscore_p"] = round(P[i].item(), 4)
        entry["bertscore_r"] = round(R[i].item(), 4)

    n = len(preds)
    bs_f = sum(e["bertscore_f"] for e in summary["per_query"]) / n
    bs_p = sum(e["bertscore_p"] for e in summary["per_query"]) / n
    bs_r = sum(e["bertscore_r"] for e in summary["per_query"]) / n

    summary["aggregate"]["bertscore_f"] = round(bs_f, 4)
    summary["aggregate"]["bertscore_p"] = round(bs_p, 4)
    summary["aggregate"]["bertscore_r"] = round(bs_r, 4)

    # Per-source BERTScore
    by_src = defaultdict(list)
    for e in summary["per_query"]:
        by_src[e["source"]].append(e["bertscore_f"])
    for src, vals in by_src.items():
        summary["by_source"][src]["bertscore_f"] = round(sum(vals)/len(vals), 4)

    print(f"BERTScore — P={bs_p:.4f}  R={bs_r:.4f}  F={bs_f:.4f}")
    return summary


if summary_main and BERTSCORE_AVAILABLE:
    summary_main = add_bertscore(summary_main)
elif not BERTSCORE_AVAILABLE:
    print("[INFO] Install bert-score and re-run this cell to compute BERTScore.")


Computing BERTScore for 59 pairs (model=distilbert-base-uncased)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore — P=0.6775  R=0.7371  F=0.7051


## 6. Evaluate All Exploration Outputs

Loops over `results/exploration/output_*.json` and ranks all prompt × parameter combinations.

In [12]:
all_summaries = {}

# Include the main output if available
if summary_main:
    all_summaries["main"] = summary_main

# Evaluate every exploration file
if EXPLORATION_DIR and os.path.isdir(EXPLORATION_DIR):
    exp_files = sorted(Path(EXPLORATION_DIR).glob("output_*.json"))
    print(f"Found {len(exp_files)} exploration output files in {EXPLORATION_DIR}")
    for exp_path in exp_files:
        tag = exp_path.stem.replace("output_", "")
        print(f"  Evaluating: {tag} ...", end=" ")
        try:
            s = evaluate_output(str(exp_path), benchmark)
            if s:
                all_summaries[tag] = s
                print(f"F1={s['aggregate']['f1']:.4f}", end="")
                if ROUGE_AVAILABLE:
                    print(f"  R1={s['aggregate']['rouge1_f']:.4f}", end="")
                print()
        except Exception as e:
            print(f"ERROR: {e}")
else:
    print(f"[SKIP] Exploration dir not found: {EXPLORATION_DIR}")

print(f"\nTotal configurations evaluated: {len(all_summaries)}")


Found 20 exploration output files in results/exploration
  Evaluating: basic_greedy ... F1=0.0789  R1=0.0969
  Evaluating: basic_t0.3_p0.9_k50 ... F1=0.0731  R1=0.0907
  Evaluating: basic_t0.7_p0.95_k100 ... F1=0.0703  R1=0.0868
  Evaluating: basic_t0.7_p0.9_k50 ... F1=0.0703  R1=0.0886
  Evaluating: basic_t1.0_p0.8_k30 ... F1=0.0736  R1=0.0932
  Evaluating: cot_greedy ... F1=0.0696  R1=0.0874
  Evaluating: cot_t0.3_p0.9_k50 ... F1=0.0677  R1=0.0838
  Evaluating: cot_t0.7_p0.95_k100 ... F1=0.0729  R1=0.0876
  Evaluating: cot_t0.7_p0.9_k50 ... F1=0.0625  R1=0.0783
  Evaluating: cot_t1.0_p0.8_k30 ... F1=0.0588  R1=0.0740
  Evaluating: enhanced_greedy ... F1=0.0745  R1=0.0918
  Evaluating: enhanced_t0.3_p0.9_k50 ... F1=0.0728  R1=0.0892
  Evaluating: enhanced_t0.7_p0.95_k100 ... F1=0.0742  R1=0.0913
  Evaluating: enhanced_t0.7_p0.9_k50 ... F1=0.0698  R1=0.0848
  Evaluating: enhanced_t1.0_p0.8_k30 ... F1=0.0696  R1=0.0832
  Evaluating: few_shot_greedy ... F1=0.0739  R1=0.0912
  Evaluating:

## 7. Leaderboard

In [13]:
if all_summaries:
    metric_keys = ["exact_match", "f1", "rouge1_f", "rouge2_f", "rougeL_f"]
    available_keys = [k for k in metric_keys
                      if k in next(iter(all_summaries.values()))["aggregate"]]

    leaderboard = []
    for tag, s in all_summaries.items():
        row = {"tag": tag}
        row.update({k: s["aggregate"][k] for k in available_keys})
        if "bertscore_f" in s["aggregate"]:
            row["bertscore_f"] = s["aggregate"]["bertscore_f"]
        leaderboard.append(row)

    # Sort by F1 (primary), ROUGE-L (secondary)
    leaderboard.sort(key=lambda x: (x.get("f1", 0), x.get("rougeL_f", 0)), reverse=True)

    # Print header
    col_w = 30
    header = f"{'Config':{col_w}s}  EM      F1      " +              ("R1      R2      RL      " if ROUGE_AVAILABLE else "") +              ("BS-F1" if any("bertscore_f" in r for r in leaderboard) else "")
    print(f"\n{'#'*90}")
    print("Leaderboard by Token F1")
    print(f"{'#'*90}")
    print(header)
    print("-" * len(header))

    for row in leaderboard:
        line = (f"{row['tag']:{col_w}s}  "
                f"{row.get('exact_match',0):.4f}  "
                f"{row.get('f1',0):.4f}  ")
        if ROUGE_AVAILABLE:
            line += (f"{row.get('rouge1_f',0):.4f}  "
                     f"{row.get('rouge2_f',0):.4f}  "
                     f"{row.get('rougeL_f',0):.4f}  ")
        if "bertscore_f" in row:
            line += f"{row['bertscore_f']:.4f}"
        print(line)



##########################################################################################
Leaderboard by Token F1
##########################################################################################
Config                          EM      F1      R1      R2      RL      BS-F1
-----------------------------------------------------------------------------
basic_greedy                    0.0000  0.0789  0.0969  0.0071  0.0748  
enhanced_greedy                 0.0000  0.0745  0.0918  0.0064  0.0684  
enhanced_t0.7_p0.95_k100        0.0000  0.0742  0.0913  0.0063  0.0670  
main                            0.0000  0.0740  0.0908  0.0048  0.0677  0.7051
few_shot_greedy                 0.0000  0.0739  0.0912  0.0061  0.0713  
basic_t1.0_p0.8_k30             0.0000  0.0736  0.0932  0.0049  0.0673  
few_shot_t0.3_p0.9_k50          0.0000  0.0733  0.0911  0.0063  0.0686  
basic_t0.3_p0.9_k50             0.0000  0.0731  0.0907  0.0061  0.0714  
cot_t0.7_p0.95_k100             0.0000  0.0729 

## 8. Save Results

In [14]:
if all_summaries:
    # Save full summary (without per_query details to keep file small)
    compact = {}
    for tag, s in all_summaries.items():
        compact[tag] = {
            "output_path": s["output_path"],
            "n_evaluated": s["n_evaluated"],
            "aggregate":   s["aggregate"],
            "by_source":   s["by_source"],
        }

    os.makedirs(os.path.dirname(EVAL_OUTPUT) or ".", exist_ok=True)
    with open(EVAL_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(compact, f, ensure_ascii=False, indent=2)
    print(f"Saved evaluation summary → {EVAL_OUTPUT}")

    # Also save full per-query details for the main/best config
    best_tag = leaderboard[0]["tag"] if leaderboard else None
    if best_tag and best_tag in all_summaries:
        detail_path = EVAL_OUTPUT.replace(".json", f"_detail_{best_tag}.json")
        with open(detail_path, "w", encoding="utf-8") as f:
            json.dump(all_summaries[best_tag], f, ensure_ascii=False, indent=2)
        print(f"Saved per-query detail    → {detail_path}")


Saved evaluation summary → results/generation_evaluation_summary.json
Saved per-query detail    → results/generation_evaluation_summary_detail_basic_greedy.json


## 9. Manual Review Helper

The cells below support the **human evaluation** dimensions from the course:
- **Correctness** — Is the answer factually accurate?
- **Faithfulness** — Is the answer grounded in the retrieved context?
- **Completeness** — Does it cover all necessary information?
- **Clarity** — Is the phrasing clear and readable?

Use the cells below to sample low-scoring queries for manual inspection.


In [ ]:
def sample_for_manual_review(summary: dict, metric: str = "f1",
                              n: int = 10, worst: bool = True) -> None:
    """
    Print the n worst (or best) queries by a given metric for manual inspection.
    Useful for identifying systematic failure modes.
    """
    if not summary or "per_query" not in summary:
        print("No per-query data available.")
        return

    sorted_entries = sorted(
        summary["per_query"],
        key=lambda e: e.get(metric, 0),
        reverse=not worst,
    )

    label = "WORST" if worst else "BEST"
    print(f"\n{'='*80}")
    print(f"  {label} {n} queries by {metric.upper()}")
    print(f"{'='*80}")

    for i, entry in enumerate(sorted_entries[:n], 1):
        score = entry.get(metric, 0)
        print(f"\n[{i}] {entry['query_id']}  {metric}={score:.4f}  source={entry['source']}")
        print(f"  Q:    {entry['query']}")
        print(f"  Gold: {entry['reference']}")
        print(f"  Pred: {entry['response'][:300]}" +
              ("..." if len(entry['response']) > 300 else ""))
        # Manual scoring template
        print(f"  ── Manual scores (fill in) ──────────────────────────────────")
        print(f"     Correctness  (0-3): ___   Faithfulness (0-3): ___")
        print(f"     Completeness (0-3): ___   Clarity      (0-3): ___")


# Show worst-scoring queries to aid manual review
if summary_main:
    sample_for_manual_review(summary_main, metric="f1", n=5, worst=True)



  WORST 5 queries by F1

[1] 4  f1=0.0000  source=Wikipedia
  Q:    What are baozi made of?
  Gold: The modern form of nigiri sushi is believed to have been invented by Hanaya Yohei, around 1824 during the Edo period.
  Pred: Baozi are made from wheat flour, typically steamed until they reach a firm consistency.
  ── Manual scores (fill in) ──────────────────────────────────
     Correctness  (0-3): ___   Faithfulness (0-3): ___
     Completeness (0-3): ___   Clarity      (0-3): ___

[2] 50  f1=0.0000  source=Wikibooks
  Q:    What is the exact calorie count of a standard bowl of tonkotsu ramen?
  Gold: After bringing to a boil, the congee is simmered over low heat for 1 hour.
  Pred: Based on the information provided in the context, a standard bowl of tonkotsu ramen contains approximately 7.9 grams of total fat.
  ── Manual scores (fill in) ──────────────────────────────────
     Correctness  (0-3): ___   Faithfulness (0-3): ___
     Completeness (0-3): ___   Clarity      (0-3): ___


## 10. Score Distribution

In [16]:
def plot_score_distribution(summary: dict, metric: str = "f1") -> None:
    """Text-based histogram of metric scores across all queries."""
    if not summary or "per_query" not in summary:
        return
    scores = [e.get(metric, 0) for e in summary["per_query"]]
    buckets = [0] * 10  # 0.0–0.1, 0.1–0.2, ...
    for s in scores:
        idx = min(int(s * 10), 9)
        buckets[idx] += 1

    print(f"\n{metric.upper()} distribution  (n={len(scores)}  "
          f"mean={sum(scores)/len(scores):.4f}  "
          f"median={sorted(scores)[len(scores)//2]:.4f})")
    print("-" * 50)
    for i, count in enumerate(buckets):
        lo, hi = i * 0.1, (i + 1) * 0.1
        bar = "█" * count
        print(f"  {lo:.1f}–{hi:.1f}  {bar:<40s} {count}")


if summary_main:
    plot_score_distribution(summary_main, "f1")
    if ROUGE_AVAILABLE:
        plot_score_distribution(summary_main, "rouge1_f")



F1 distribution  (n=59  mean=0.0740  median=0.0606)
--------------------------------------------------
  0.0–0.1  ██████████████████████████████████████████████ 46
  0.1–0.2  ██████████                               10
  0.2–0.3  ███                                      3
  0.3–0.4                                           0
  0.4–0.5                                           0
  0.5–0.6                                           0
  0.6–0.7                                           0
  0.7–0.8                                           0
  0.8–0.9                                           0
  0.9–1.0                                           0

ROUGE1_F distribution  (n=59  mean=0.0908  median=0.0741)
--------------------------------------------------
  0.0–0.1  ████████████████████████████████████████ 40
  0.1–0.2  ███████████████                          15
  0.2–0.3  ████                                     4
  0.3–0.4                                           0
  0.4–0.5           